# Held-out model evaluation

Train first with `uv run pm-train`. This notebook regenerates predictions and reports capped-RUL metrics at the final recorded cycle of each test engine. Install the notebooks extra for plotting.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
import pandas as pd
from predictive_maintenance.config import MODELS_DIR, PROCESSED_DATA_DIR, RESULTS_DIR
from predictive_maintenance.models.predict import predict
from predictive_maintenance.models.evaluate import evaluate, align_predictions

predictions_path = PROCESSED_DATA_DIR / "test_predictions.csv"
labels_path = PROCESSED_DATA_DIR / "test_labels.csv"
predict(MODELS_DIR / "best_model.json", PROCESSED_DATA_DIR / "test_features.csv", predictions_path)
evaluate(predictions_path, labels_path, RESULTS_DIR / "evaluation.csv", RESULTS_DIR / "evaluation_by_unit.csv")
pd.read_csv(RESULTS_DIR / "evaluation.csv")

In [ ]:
import matplotlib.pyplot as plt

joined = align_predictions(pd.read_csv(predictions_path), pd.read_csv(labels_path))
last = joined.loc[joined.groupby("unit")["cycle"].idxmax()]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(last.RUL, last.prediction, alpha=0.7)
limit = max(last.RUL.max(), last.prediction.max())
ax.plot([0, limit], [0, limit], "--", color="black")
ax.set(xlabel="True capped RUL (cycles)", ylabel="Predicted RUL (cycles)", title="Final cycle of each test engine")
plt.show()